In [1]:
import scipy.optimize as opt
import numpy as np
import xarray as xr
import opt_function as opt_func
from pathlib import Path
module_dir = Path("/glade/work/iranjan/tpose24-osse/")
sys.path.append(str(module_dir))
import osse_tools as ost

In [2]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [3]:
TRUE_W = "/glade/work/iranjan/fast-osse/fastosse-run1/true_w.nc"
PMO_SCRIPT = "/glade/work/iranjan/DART/models/MOM6/work/perfect_model_obs"
true_w = xr.open_dataset(TRUE_W)

In [5]:
lat_bounds = (-1.0, 2.0)
lon_bounds = (218.5, 221.5)

lower_bounds = []
upper_bounds = []
for _ in range(6):
    lower_bounds.extend([lat_bounds[0], lon_bounds[0]])
    upper_bounds.extend([lat_bounds[1], lon_bounds[1]])

bounds = opt.Bounds(lower_bounds, upper_bounds)

# --- Generate Initial Guess (Spread evenly across the box) ---
# Instead of random initialization, we space them out to prevent initial collisions
initial_lats = np.linspace(lat_bounds[0] + 0.3, lat_bounds[1] - 0.3, 6)
initial_lons = np.linspace(lon_bounds[0] + 0.3, lon_bounds[1] - 0.3, 6)

x0 = np.empty(12)
x0[0::2] = initial_lats
x0[1::2] = initial_lons

# --- Scipy Objective Wrapper Function with Distance Penalty ---
def objective_function(x, true_w_dataset, min_dist_deg=0.15):
    """
    Wrapper that unpacks glider positions, checks for spatial overlap,
    and returns a penalty or executes opt_loc.
    """
    # Reshape into a 2D numpy array: shape (6, 2) -> columns: [lat, lon]
    loc_matrix = x.reshape(6, 2)
    loc_list = loc_matrix.tolist()
    
    # 1. CRITICAL: Check distance between every single pair of gliders
    too_close = False
    closest_distance = float('inf')
    
    for i in range(6):
        for j in range(i + 1, 6):
            # Calculate standard Euclidean distance in degrees
            dist = np.sqrt((loc_matrix[i, 0] - loc_matrix[j, 0])**2 + 
                           (loc_matrix[i, 1] - loc_matrix[j, 1])**2)
            if dist < closest_distance:
                closest_distance = dist
            if dist < min_dist_deg:
                too_close = True

    # 2. Penalty logic if gliders collapse onto the same coordinates
    if too_close:
        print(f"--> SKIPPING RUN (Gliders too close! Closest pair distance: {closest_distance:.4f}°)")
        # Return a massive penalty scaled by how badly they violated the distance rules
        return 1e6 + (min_dist_deg - closest_distance) * 1e5

    """print(f"\n--- Evaluating Glider Locations ---")
    for idx, (lat, lon) in enumerate(loc_list):
        print(f" Glider {idx+1}: Lat={lat:.4f}, Lon={lon:.4f}")"""
        
    # 3. Call your standard simulation pipeline
    try:
        norm_error = opt_func.opt_loc(loc_list, true_w_dataset)
        print(f"--> Resulting L2 Norm Error: {norm_error:.6e}")
        return norm_error
    except Exception as e:
        print(f"--> Warning: Simulation failed for this configuration: {e}")
        return 1e10 


# --- Run the Minimization ---

res = opt.minimize(
    fun=objective_function,
    x0=x0,
    args=(true_w, 0.15),  # Passes true_w and a 0.15-degree minimum distance
    bounds=bounds,
    method='Powell', 
    options={'disp': True, 'maxiter': 20}
)

# --- Extract and print results ---
best_loc_list = res.x.reshape(6, 2).tolist()
print("\n================ OPTIMIZATION COMPLETE ================")
print(f"Optimization Status: {res.message}")
print(f"Minimum Error Norm Reached: {res.fun}")
print("Best Glider Tracking Coordinates Found:")
for idx, (lat, lon) in enumerate(best_loc_list):
    print(f" Glider {idx+1}: Lat={lat:.6f}, Lon={lon:.6f}")


Found 8 unique date groups spanning 2015-09-04 00:00:00 to 2015-10-25 00:00:00.
Found 8 subdirectories, running with MAX_CONCURRENT=128
SUCCESS: EEP_MITgcm185Lvgrid_Whitt2026hgrid.mom6.h.z.2015-10-025
SUCCESS: EEP_MITgcm185Lvgrid_Whitt2026hgrid.mom6.h.z.2015-09-018
SUCCESS: EEP_MITgcm185Lvgrid_Whitt2026hgrid.mom6.h.z.2015-10-004
SUCCESS: EEP_MITgcm185Lvgrid_Whitt2026hgrid.mom6.h.z.2015-09-025
SUCCESS: EEP_MITgcm185Lvgrid_Whitt2026hgrid.mom6.h.z.2015-10-011
SUCCESS: EEP_MITgcm185Lvgrid_Whitt2026hgrid.mom6.h.z.2015-10-018
SUCCESS: EEP_MITgcm185Lvgrid_Whitt2026hgrid.mom6.h.z.2015-09-011
SUCCESS: EEP_MITgcm185Lvgrid_Whitt2026hgrid.mom6.h.z.2015-09-004
All done. 0 failure(s).

Found 8 obs_seq.out files
Total obs in joined sequence: 3552
Inferred cadence: 7 days 00:00:00. Snapped timestamps to this grid (max shift applied: 2 days 00:00:00).
Found 6 distinct glider positions.
--> Resulting L2 Norm Error: 2.304961e+10
Found 8 unique date groups spanning 2015-09-04 00:00:00 to 2015-10-25 00:00:

KeyboardInterrupt: 